In [23]:
"""
Optimized pipeline for large-scale climate report analysis
Features:
- Pre-filtering with climate detector
- Batch processing
- Caching preprocessed data
- Multi-stage analysis
- Multi-language support with translation
"""


import json
from pathlib import Path
from typing import List, Dict, Tuple
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm
import langid

class ClimateReportAnalyzer:
    # Configuration constants
    MIN_CHUNK_LENGTH = 200  # Minimum characters for a chunk to be analyzed
    BATCH_SIZE = 48         # Number of chunks to process in parallel

    def __init__(self, cache_dir="cache", use_cache=True, auto_translate=False,
                 run_climate_filter=True, run_specificity=True,
                 run_sentiment=False, run_commitment=False):
        """
        Initialize analyzer with configuration

        Args:
            cache_dir: Directory for caching results
            use_cache: Use cached text extractions
            auto_translate: Automatically translate non-English texts
            run_climate_filter: Pre-filter with climate detector
            run_specificity: Run specificity analysis
            run_sentiment: Run sentiment analysis
            run_commitment: Run commitment detection
        """
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        # Store config
        self.use_cache = use_cache
        self.auto_translate = auto_translate
        self.run_climate_filter = run_climate_filter
        self.run_specificity = run_specificity
        self.run_sentiment = run_sentiment
        self.run_commitment = run_commitment

        self.device = 0 if torch.cuda.is_available() else -1
        print(f"Using device: {'GPU' if self.device == 0 else 'CPU'}")
        print(f"Config: cache={use_cache}, translate={auto_translate}, filter={run_climate_filter}, "
              f"spec={run_specificity}, sent={run_sentiment}, commit={run_commitment}")

        self.models = {}
        self.translators = {}

        # Helsinki-NLP translation models for common languages
        self.supported_languages = {
            'de': 'Helsinki-NLP/opus-mt-de-en',  # German
            'fr': 'Helsinki-NLP/opus-mt-fr-en',  # French
            'es': 'Helsinki-NLP/opus-mt-es-en',  # Spanish
            'it': 'Helsinki-NLP/opus-mt-it-en',  # Italian
            'pt': 'Helsinki-NLP/opus-mt-tc-big-pt-en',  # Portuguese
            'nl': 'Helsinki-NLP/opus-mt-nl-en',  # Dutch
            'pl': 'Helsinki-NLP/opus-mt-pl-en',  # Polish
            'sv': 'Helsinki-NLP/opus-mt-sv-en',  # Swedish
            'da': 'Helsinki-NLP/opus-mt-da-en',  # Danish
            'fi': 'Helsinki-NLP/opus-mt-fi-en',  # Finnish
            'no': 'Helsinki-NLP/opus-mt-no-en',  # Norwegian
            'ru': 'Helsinki-NLP/opus-mt-ru-en',  # Russian
            'zh': 'Helsinki-NLP/opus-mt-zh-en',  # Chinese
            'ja': 'Helsinki-NLP/opus-mt-ja-en',  # Japanese
            'ko': 'Helsinki-NLP/opus-mt-ko-en',  # Korean
        }

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text from PDF with caching"""
        cache_file = self.cache_dir / f"{Path(pdf_path).stem}_text.txt"

        if self.use_cache and cache_file.exists():
            print(f"Loading cached text from {cache_file}")
            return cache_file.read_text(encoding='utf-8')

        print(f"Extracting text from {pdf_path}...")
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        if self.use_cache:
            cache_file.write_text(text, encoding='utf-8')
            print(f"Cached text to {cache_file}")

        return text

    def chunk_text_by_paragraphs(self, text: str, max_tokens=400) -> List[str]:
        """
        Split by paragraphs, merge small ones, split large ones

        Args:
            max_tokens: Maximum tokens per chunk (default 400, model max is 512)

        Returns:
            List of text chunks, each roughly paragraph-sized
        """
        # Rough estimate: 1 token ≈ 4 characters
        MAX_CHARS = max_tokens * 4  # 400 tokens ≈ 1600 chars

        # Step 1: Split by double newline (paragraphs)
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            # Case A: Paragraph is way too big → split it by sentences
            if para_size > MAX_CHARS:
                # Save current chunk first
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                # Split large paragraph into sentences
                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            # Case B: Adding this paragraph would exceed limit → save & start new
            elif current_size + para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            # Case C: Small paragraph → accumulate
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        # Don't forget last chunk
        if current_chunk:
            chunks.append(current_chunk.strip())

        # Filter very short chunks
        return [c for c in chunks if len(c) > self.MIN_CHUNK_LENGTH]

    def detect_language(self, text: str) -> str:
        """
        Detect language of text using langid

        Args:
            text: Text sample to detect language from

        Returns:
            ISO 639-1 language code (e.g., 'en', 'de', 'es')
        """
        try:
            # Use first 5000 chars for detection (more reliable)
            sample = text[:5000]
            lang, confidence = langid.classify(sample)
            print(f"Language detection: {lang} (confidence: {confidence:.3f})")
            return lang
        except Exception as e:
            print(f"Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def load_translator(self, lang: str):
        """Lazy load translation model for given language"""
        if lang not in self.supported_languages:
            raise ValueError(f"Translation not supported for language: {lang}. "
                           f"Supported: {list(self.supported_languages.keys())}")

        if lang not in self.translators:
            print(f"Loading translation model for {lang}→en...")
            model_name = self.supported_languages[lang]
            self.translators[lang] = pipeline(
                "translation",
                model=model_name,
                device=self.device,
                max_length=512
            )

        return self.translators[lang]

    def translate_to_english(self, chunks: List[str], lang: str,
                            batch_size=None) -> Tuple[List[str], str]:
        """
        Translate chunks from source language to English

        Args:
            chunks: List of text chunks in source language
            lang: Source language code
            batch_size: Batch size for translation

        Returns:
            tuple: (translated_chunks, cache_suffix)
        """
        if batch_size is None:
            batch_size = 8  # Smaller batch for translation (memory intensive)

        # Check cache first
        cache_suffix = f"_translated_{lang}_en"

        translator = self.load_translator(lang)
        translated_chunks = []

        print(f"\nTranslating {len(chunks)} chunks from {lang} to English...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
            batch = chunks[i:i+batch_size]
            try:
                # Translation models expect list of strings
                results = translator(batch, batch_size=batch_size)

                for result in results:
                    # Extract translated text
                    if isinstance(result, dict):
                        translated_text = result.get('translation_text', '')
                    elif isinstance(result, list) and len(result) > 0:
                        translated_text = result[0].get('translation_text', '')
                    else:
                        translated_text = str(result)

                    translated_chunks.append(translated_text)

            except Exception as e:
                print(f"\nTranslation error in batch {i//batch_size}: {e}")
                # Fallback: keep original text for failed translations
                translated_chunks.extend(batch)
                continue

        print(f"Translation complete: {len(translated_chunks)} chunks")
        return translated_chunks, cache_suffix

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter chunks to keep only climate-related ones"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        # Process in batches for MUCH better performance (10-20x speedup)
        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting climate content"):
            batch = chunks[i:i+batch_size]
            try:
                # KEY: Pass batch_size to enable batching in the pipeline
                results = detector(batch, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    # Keep if labeled as "climate" or similar
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"Kept {len(climate_chunks)} climate-related chunks ({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks

    def analyze_specificity(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate specificity with batching for performance"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity",
            "specificity"
        )

        print(f"\nAnalyzing specificity for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing specificity"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                # BATCHED: 10-20x faster than processing one-by-one
                results = specificity(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_sentiment(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate sentiment (opportunity/neutral/risk)"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment",
            "sentiment"
        )

        print(f"\nAnalyzing sentiment for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing sentiment"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = sentiment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_commitments(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Detect climate commitments and actions"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment",
            "commitment"
        )

        print(f"\nAnalyzing commitments for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing commitments"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = commitment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def calculate_report_score(self, analyzed_chunks: List[Dict]) -> Dict:
        """Calculate overall scores for the report"""
        # Correct labels for climate-specificity model
        score_map = {
            "specific": 1.0,
            "non-specific": 0.0,
        }

        specificity_scores = []
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', '').lower()
            confidence = chunk.get('specificity_score', 0.5)

            # Map label to score
            base_score = score_map.get(label, 0.5)  # Default 0.5 if unknown
            weighted_score = base_score * confidence
            specificity_scores.append(weighted_score)

        overall_score = sum(specificity_scores) / len(specificity_scores) if specificity_scores else 0

        # Count labels
        label_dist = {}
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', 'unknown')
            label_dist[label] = label_dist.get(label, 0) + 1

        return {
            'overall_specificity': overall_score,
            'num_chunks_analyzed': len(analyzed_chunks),
            'num_climate_chunks': len([c for c in analyzed_chunks if c.get('specificity_label') == 'specific']),
            'label_distribution': label_dist,
            'specificity_scores': specificity_scores
        }

    def process_report(self, pdf_path: str):
        """
        Full pipeline for processing a report
        Uses config flags set in __init__ to determine which analyses to run
        """
        print(f"\n{'='*80}")
        print(f"Processing: {pdf_path}")
        print(f"{'='*80}")

        pdf_stem = Path(pdf_path).stem

        # Step 1: Extract text
        text = self.extract_text_from_pdf(pdf_path)
        print(f"Total text length: {len(text)} characters")

        # Step 2: Detect language and translate if needed
        lang_cache = self.cache_dir / f"{pdf_stem}_lang.txt"

        if self.use_cache and lang_cache.exists():
            lang = lang_cache.read_text(encoding='utf-8').strip()
            print(f"Loaded cached language: {lang}")
        else:
            lang = self.detect_language(text)
            if self.use_cache:
                lang_cache.write_text(lang, encoding='utf-8')

        # Step 3: Chunk text (by paragraphs)
        chunks = self.chunk_text_by_paragraphs(text)
        print(f"Created {len(chunks)} chunks")

        # Step 4: Translate if not English and auto_translate enabled
        cache_suffix = ""
        if lang != 'en' and self.auto_translate:
            if lang not in self.supported_languages:
                print(f"⚠️  Translation not supported for language: {lang}")
                print(f"   Supported languages: {list(self.supported_languages.keys())}")
                print(f"   Proceeding with original text (may affect analysis accuracy)")
                cache_suffix = f"_unsupported_{lang}"
            else:
                trans_cache = self.cache_dir / f"{pdf_stem}_translated_{lang}_en.json"

                if self.use_cache and trans_cache.exists():
                    print(f"Loading cached translation from {trans_cache}")
                    with open(trans_cache, 'r', encoding='utf-8') as f:
                        chunks = json.load(f)
                    cache_suffix = f"_translated_{lang}_en"
                else:
                    chunks, cache_suffix = self.translate_to_english(chunks, lang)

                    if self.use_cache:
                        with open(trans_cache, 'w', encoding='utf-8') as f:
                            json.dump(chunks, f, ensure_ascii=False, indent=2)
                        print(f"Cached translation to {trans_cache}")

        # Step 5: Filter for climate content (optional)
        if self.run_climate_filter:
            climate_chunks = self.filter_climate_chunks(chunks)
        else:
            climate_chunks = [{'text': c} for c in chunks]

        # Step 6-8: Run enabled analyses
        analyzed_chunks = climate_chunks

        if self.run_specificity:
            analyzed_chunks = self.analyze_specificity(analyzed_chunks)

        if self.run_sentiment:
            analyzed_chunks = self.analyze_sentiment(analyzed_chunks)

        if self.run_commitment:
            analyzed_chunks = self.analyze_commitments(analyzed_chunks)

        # Step 9: Calculate scores
        scores = self.calculate_report_score(analyzed_chunks)

        # Step 10: Save results
        results = {
            'pdf_path': str(pdf_path),
            'language': lang,
            'translation': cache_suffix if cache_suffix else 'none',
            'total_chunks': len(chunks),
            'climate_chunks': len(climate_chunks),
            'scores': scores,
            'sample_chunks': analyzed_chunks[:10]  # Save first 10 for inspection
        }

        results_file = self.cache_dir / f"{pdf_stem}_results.json"
        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"\nSaved results to {results_file}")

        return scores

    def process_all_reports(self, reports_dir: str, pattern="**/*.pdf"):
        """Process all PDFs in directory tree using config from __init__"""
        pdf_files = list(Path(reports_dir).glob(pattern))
        print(f"\nFound {len(pdf_files)} PDF files in {reports_dir}")

        results = []
        failed = []

        for pdf_path in tqdm(pdf_files, desc="Processing reports"):
            try:
                scores = self.process_report(str(pdf_path))
                results.append({
                    'file': str(pdf_path),
                    'company': pdf_path.parent.name,
                    'year': self._extract_year(pdf_path.name),
                    'scores': scores
                })
            except Exception as e:
                print(f"\n❌ Failed: {pdf_path}")
                print(f"   Error: {e}")
                failed.append(str(pdf_path))
                continue

        # Print summary
        self._print_batch_summary(results, failed, len(pdf_files))

        # Save aggregated results
        output_file = self.cache_dir / 'all_results.json'
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump({
                'results': results,
                'failed': failed,
                'total_processed': len(results),
                'total_found': len(pdf_files)
            }, f, indent=2, ensure_ascii=False)
        print(f"Results saved to: {output_file}")

        return results

    def process(self, path: str):
        """
        Smart processor: automatically detects if path is a file or directory
        Uses config from __init__ for all processing options
        """
        path_obj = Path(path)

        if not path_obj.exists():
            raise FileNotFoundError(f"Path not found: {path}")

        # Case 1: Single PDF file
        if path_obj.is_file() and path_obj.suffix.lower() == '.pdf':
            print(f"\n{'='*80}")
            print(f"PROCESSING SINGLE PDF")
            print(f"{'='*80}")

            scores = self.process_report(str(path_obj))
            self._print_single_summary(path_obj, scores)
            return scores

        # Case 2: Directory
        elif path_obj.is_dir():
            nested_pdfs = list(path_obj.glob("**/*.pdf"))

            if not nested_pdfs:
                print(f"❌ No PDF files found in {path}")
                return []

            print(f"\n{'='*80}")
            print(f"PROCESSING DIRECTORY: {path}")
            print(f"{'='*80}")
            print(f"Found {len(nested_pdfs)} PDF files")

            results = self.process_all_reports(str(path_obj), pattern="**/*.pdf")
            return results

        else:
            raise ValueError(f"Path must be a PDF file or directory: {path}")

    def _extract_year(self, filename: str) -> str:
        """Extract year from filename (e.g., Report_2020.pdf -> 2020)"""
        import re
        match = re.search(r'20\d{2}', filename)
        return match.group(0) if match else 'unknown'

    def _print_single_summary(self, pdf_path: Path, scores: Dict):
        """Pretty print results for single PDF"""
        print("\n" + "="*80)
        print("FINAL RESULTS")
        print("="*80)
        print(f"File: {pdf_path.name}")
        print(f"Company: {pdf_path.parent.name}")

        print(f"\n📊 Component Scores:")
        if 'overall_specificity' in scores:
            print(f"  Specificity: {scores['overall_specificity']:.3f} - {self._interpret_score(scores['overall_specificity'])}")

        print(f"\nChunks analyzed: {scores['num_chunks_analyzed']}")

        # Show distributions
        if 'label_distribution' in scores:
            print(f"\nLabel Distribution:")
            for label, count in sorted(scores['label_distribution'].items()):
                pct = count / scores['num_chunks_analyzed'] * 100
                bar = '█' * int(pct / 2)
                print(f"  {label:15s}: {count:3d} ({pct:5.1f}%) {bar}")

    def _print_batch_summary(self, results: List[Dict], failed: List[str], total: int):
        """Pretty print results for batch processing"""
        print(f"\n{'='*80}")
        print(f"BATCH PROCESSING SUMMARY")
        print(f"{'='*80}")
        print(f"✅ Successfully processed: {len(results)}/{total}")

        if failed:
            print(f"❌ Failed: {len(failed)}")
            for f in failed[:5]:  # Show first 5
                print(f"   - {Path(f).name}")
            if len(failed) > 5:
                print(f"   ... and {len(failed)-5} more")

        if results:
            # Calculate aggregate statistics
            avg_score = sum(r['scores']['overall_specificity'] for r in results) / len(results)
            print(f"\nAggregate Statistics:")
            print(f"  Average Specificity Score: {avg_score:.3f}")
            print(f"  → {self._interpret_score(avg_score)}")

            # Top 5 and bottom 5
            sorted_results = sorted(results, key=lambda x: x['scores']['overall_specificity'], reverse=True)

            print(f"\n📊 Top 5 Most Specific Reports:")
            for i, r in enumerate(sorted_results[:5], 1):
                score = r['scores']['overall_specificity']
                print(f"  {i}. {r['company']:20s} ({r.get('year', '?')}): {score:.3f}")

            print(f"\n📊 Top 5 Least Specific Reports:")
            for i, r in enumerate(sorted_results[-5:], 1):
                score = r['scores']['overall_specificity']
                print(f"  {i}. {r['company']:20s} ({r.get('year', '?')}): {score:.3f}")

    def _interpret_score(self, score: float) -> str:
        """Interpret specificity score"""
        if score >= 0.7:
            return "Highly specific (concrete targets & actions)"
        elif score >= 0.5:
            return "Moderately specific (mix of concrete & vague)"
        elif score >= 0.3:
            return "Low specificity (mostly vague statements)"
        else:
            return "Very low specificity (lacks concrete information)"

    def inspect_chunks_interactive(self, pdf_path: str, mode="random", n=3,
                                  min_confidence=0.0, max_confidence=1.0,
                                  label_filter=None):
        """
        Advanced chunk inspection for manual evaluation

        Args:
            pdf_path: Path to PDF
            mode: "random", "high_conf", "low_conf", "high_score", "low_score"
            n: Number of chunks to show
            min_confidence: Minimum confidence threshold
            max_confidence: Maximum confidence threshold
            label_filter: Filter by label (e.g., "specific", "opportunity")
        """
        import random

        # Load results
        results_file = self.cache_dir / f"{Path(pdf_path).stem}_results.json"
        if not results_file.exists():
            print(f"No results found. Run process() first.")
            return

        with open(results_file, encoding='utf-8') as f:
            data = json.load(f)

        chunks = data.get('sample_chunks', [])
        if not chunks:
            print("No chunks available. Re-run analysis.")
            return

        # Filter by label if specified
        if label_filter:
            chunks = [c for c in chunks if label_filter.lower() in
                     str(c.get('specificity_label', '')).lower() or
                     label_filter.lower() in str(c.get('sentiment_label', '')).lower()]

        # Filter by confidence range
        chunks = [c for c in chunks if min_confidence <= c.get('specificity_score', 0.5) <= max_confidence]

        if not chunks:
            print(f"No chunks matching criteria (label={label_filter}, conf={min_confidence}-{max_confidence})")
            return

        # Select chunks based on mode
        if mode == "random":
            selected = random.sample(chunks, min(n, len(chunks)))
        elif mode == "high_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 0), reverse=True)
            selected = sorted_chunks[:n]
        elif mode == "low_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 1))
            selected = sorted_chunks[:n]
        else:
            selected = chunks[:n]

        # Display
        print(f"\n{'='*80}")
        print(f"CHUNK INSPECTION - Mode: {mode}")
        print(f"PDF: {Path(pdf_path).name}")
        print(f"Showing {len(selected)} of {len(chunks)} chunks")
        print(f"{'='*80}\n")

        for i, chunk in enumerate(selected, 1):
            print(f"{'─'*80}")
            print(f"Chunk #{i}")
            print(f"{'─'*80}")

            # Show all scores
            spec_label = chunk.get('specificity_label', 'N/A')
            spec_conf = chunk.get('specificity_score', 0)
            print(f"Specificity: {spec_label} (confidence: {spec_conf:.3f})")

            if 'sentiment_label' in chunk:
                sent_label = chunk.get('sentiment_label')
                sent_conf = chunk.get('sentiment_score', 0)
                print(f"Sentiment:   {sent_label} (confidence: {sent_conf:.3f})")

            if 'commitment_label' in chunk:
                commit_label = chunk.get('commitment_label')
                commit_conf = chunk.get('commitment_score', 0)
                print(f"Commitment:  {commit_label} (confidence: {commit_conf:.3f})")

            if 'detector_score' in chunk:
                det_conf = chunk.get('detector_score')
                print(f"Climate relevance: {det_conf:.3f}")

            text = chunk.get('text', '')
            print(f"\nText ({len(text)} chars):")
            print(f"{text[:500]}{'...' if len(text) > 500 else ''}\n")

    def check_translations(self, reports_dir: str = None):
        """
        Check language detection and translation status for all cached PDFs

        Args:
            reports_dir: Directory to scan (optional, uses cache if None)
        """
        print(f"\n{'='*80}")
        print(f"TRANSLATION STATUS CHECK")
        print(f"{'='*80}\n")

        # Find all result files
        result_files = list(self.cache_dir.glob("*_results.json"))

        if not result_files:
            print("No cached results found. Process some PDFs first.")
            return

        stats = {
            'en': 0,
            'translated': 0,
            'unsupported': 0,
            'by_language': {}
        }

        print(f"{'PDF':<40} {'Lang':<6} {'Status':<30}")
        print(f"{'-'*80}")

        for result_file in sorted(result_files):
            try:
                with open(result_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                pdf_name = Path(data.get('pdf_path', result_file.stem)).name[:38]
                lang = data.get('language', 'unknown')
                translation = data.get('translation', 'unknown')

                # Update stats
                if lang not in stats['by_language']:
                    stats['by_language'][lang] = 0
                stats['by_language'][lang] += 1

                if lang == 'en':
                    stats['en'] += 1
                    status_icon = "✓"
                elif 'translated' in translation:
                    stats['translated'] += 1
                    status_icon = "🔄"
                elif 'unsupported' in translation:
                    stats['unsupported'] += 1
                    status_icon = "⚠️ "
                else:
                    status_icon = "?"

                print(f"{pdf_name:<40} {lang:<6} {status_icon} {translation}")

            except Exception as e:
                print(f"{result_file.name:<40} ERROR: {e}")

        # Summary
        print(f"\n{'-'*80}")
        print(f"SUMMARY")
        print(f"{'-'*80}")
        total = len(result_files)
        print(f"Total PDFs processed: {total}")
        print(f"  ✓ English (no translation): {stats['en']} ({stats['en']/total*100:.1f}%)")
        print(f"  🔄 Translated: {stats['translated']} ({stats['translated']/total*100:.1f}%)")
        print(f"  ⚠️  Unsupported: {stats['unsupported']} ({stats['unsupported']/total*100:.1f}%)")

        print(f"\nLanguage breakdown:")
        for lang, count in sorted(stats['by_language'].items(), key=lambda x: x[1], reverse=True):
            print(f"  {lang}: {count}")

    def inspect_translation(self, pdf_path: str, n_samples=3):
        """
        Show original and translated text side-by-side for validation

        Args:
            pdf_path: Path to PDF
            n_samples: Number of chunk pairs to show
        """
        pdf_stem = Path(pdf_path).stem

        # Load original text and language
        orig_cache = self.cache_dir / f"{pdf_stem}_text.txt"
        lang_file = self.cache_dir / f"{pdf_stem}_lang.txt"

        if not orig_cache.exists() or not lang_file.exists():
            print(f"No cached data found for {pdf_path}")
            return

        orig_text = orig_cache.read_text(encoding='utf-8')
        lang = lang_file.read_text(encoding='utf-8').strip()

        if lang == 'en':
            print(f"PDF is in English, no translation needed.")
            return

        # Load translated chunks
        trans_cache = self.cache_dir / f"{pdf_stem}_translated_{lang}_en.json"
        if not trans_cache.exists():
            print(f"No translation found. Run process() with auto_translate=True first.")
            return

        with open(trans_cache, 'r', encoding='utf-8') as f:
            translated_chunks = json.load(f)

        # Get original chunks
        orig_chunks = self.chunk_text_by_paragraphs(orig_text)

        print(f"\n{'='*80}")
        print(f"TRANSLATION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {lang} → en")
        print(f"{'='*80}\n")

        # Show random samples
        import random
        indices = random.sample(range(min(len(orig_chunks), len(translated_chunks))),
                               min(n_samples, len(orig_chunks), len(translated_chunks)))

        for i, idx in enumerate(indices, 1):
            orig = orig_chunks[idx]
            trans = translated_chunks[idx] if idx < len(translated_chunks) else "N/A"

            print(f"{'─'*80}")
            print(f"Sample {i} (chunk #{idx})")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({lang}):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")
            print(f"\nTRANSLATED (en):")
            print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}\n")

    def show_chunk_structure(self, pdf_path: str, stage="all"):
        """
        Show chunk structure at different pipeline stages

        Args:
            pdf_path: Path to PDF
            stage: "raw", "filtered", "analyzed", or "all"
        """
        print(f"\n{'='*80}")
        print(f"CHUNK STRUCTURE ANALYSIS")
        print(f"{'='*80}\n")

        # Extract and chunk
        text = self.extract_text_from_pdf(pdf_path)
        raw_chunks = self.chunk_text_by_paragraphs(text)

        if stage in ["raw", "all"]:
            print(f"STAGE 1: Raw chunks (after text splitting)")
            print(f"  Type: List[str]")
            print(f"  Count: {len(raw_chunks)}")
            print(f"  Example structure:")
            if raw_chunks:
                print(f"    chunk[0] = {repr(raw_chunks[0][:100])}...")
                print(f"    Length: {len(raw_chunks[0])} chars\n")

        if stage in ["filtered", "analyzed", "all"]:
            # Filter climate
            if self.run_climate_filter:
                filtered_chunks = self.filter_climate_chunks(raw_chunks)
            else:
                filtered_chunks = [{'text': c} for c in raw_chunks]

            print(f"STAGE 2: Filtered chunks (after climate detection)")
            print(f"  Type: List[Dict]")
            print(f"  Count: {len(filtered_chunks)}")
            print(f"  Example structure:")
            if filtered_chunks:
                print(f"    chunk[0] = {{")
                print(f"      'text': {repr(filtered_chunks[0]['text'][:80])}...,")
                print(f"      'detector_score': {filtered_chunks[0].get('detector_score', 'N/A')}")
                print(f"    }}\n")

        if stage in ["analyzed", "all"]:
            # Analyze
            if self.run_specificity:
                analyzed_chunks = self.analyze_specificity(filtered_chunks[:5])  # Just 5 for demo
            else:
                analyzed_chunks = filtered_chunks[:5]

            print(f"STAGE 3: Analyzed chunks (after specificity)")
            print(f"  Type: List[Dict]")
            print(f"  Example structure:")
            if analyzed_chunks:
                print(f"    chunk[0] = {{")
                print(f"      'text': {repr(analyzed_chunks[0]['text'][:60])}...,")
                print(f"      'detector_score': {analyzed_chunks[0].get('detector_score', 'N/A')},")
                print(f"      'specificity_label': {analyzed_chunks[0].get('specificity_label', 'N/A')},")
                print(f"      'specificity_score': {analyzed_chunks[0].get('specificity_score', 'N/A')}")
                if self.run_sentiment:
                    print(f"      'sentiment_label': {analyzed_chunks[0].get('sentiment_label', 'N/A')},")
                    print(f"      'sentiment_score': {analyzed_chunks[0].get('sentiment_score', 'N/A')}")
                print(f"    }}\n")


# ============================================================================
# Usage
# ============================================================================

if __name__ == "__main__":
    # === TESTING TRANSLATION PIPELINE ===

    # Step 1: Test on German PDF
    analyzer = ClimateReportAnalyzer(
        cache_dir="cache",
        use_cache=True,
        auto_translate=True,        # Enable translation
        run_climate_filter=False,   # Disable for speed during testing
        run_specificity=False,      # Disable for speed during testing
        run_sentiment=False,
        run_commitment=False
    )

    # Test on German PDF
    pdf_path = "data/reports/Dillinger/012_2015_annual_report.pdf"
    scores = analyzer.process(pdf_path)

    # Inspect translation quality
    print("\n" + "="*80)
    print("Checking translation quality:")
    analyzer.inspect_translation(pdf_path, n_samples=3)

    # Step 2: Check all PDFs in directory (after testing single file)
    # Process all first
    # analyzer.process("data/reports")

    # Then check translation status
    # analyzer.check_translations()

    # === FULL PIPELINE (after translation tested and verified) ===
    # analyzer = ClimateReportAnalyzer(
    #     use_cache=True,
    #     auto_translate=True,
    #     run_climate_filter=True,
    #     run_specificity=True,
    #     run_sentiment=True,
    #     run_commitment=True
    # )
    # results = analyzer.process("data/reports")

    # === VALIDATION & INSPECTION ===
    # analyzer.inspect_chunks_interactive(pdf_path, mode="random", n=3)
    # analyzer.show_chunk_structure(pdf_path, stage="all")

Using device: CPU
Config: cache=True, translate=True, filter=False, spec=False, sent=False, commit=False

PROCESSING SINGLE PDF

Processing: data/reports/Dillinger/012_2015_annual_report.pdf
Extracting text from data/reports/Dillinger/012_2015_annual_report.pdf...


Reading pages:   0%|          | 0/70 [00:00<?, ?it/s]

Cached text to cache/012_2015_annual_report_text.txt
Total text length: 142136 characters
Language detection: de (confidence: -13385.168)
Created 12 chunks
Loading translation model for de→en...


Device set to use cpu



Translating 12 chunks from de to English...


Translating:   0%|          | 0/2 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (575 > 512). Running this sequence through the model will result in indexing errors
Your input_length: 19490 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 847 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)



Translation error in batch 0: index out of range in self

Translation error in batch 1: index out of range in self
Translation complete: 12 chunks
Cached translation to cache/012_2015_annual_report_translated_de_en.json

Saved results to cache/012_2015_annual_report_results.json

FINAL RESULTS
File: 012_2015_annual_report.pdf
Company: Dillinger

📊 Component Scores:
  Specificity: 0.250 - Very low specificity (lacks concrete information)

Chunks analyzed: 12

Label Distribution:
  unknown        :  12 (100.0%) ██████████████████████████████████████████████████

Checking translation quality:

TRANSLATION INSPECTION: 012_2015_annual_report.pdf
Language: de → en

────────────────────────────────────────────────────────────────────────────────
Sample 1 (chunk #3)
────────────────────────────────────────────────────────────────────────────────

ORIGINAL (de):
R.
JÜRGEN BLUDAU, Dillingen HENNER WITTLING, Ottweiler
Mitglied des Konzernbetriebsrats und stellvertretender Mitglied des Vorstands 